# Ajuster la voix pidgin sur Common Voice camerounais (Colab)

La voix de base (MMS-TTS pcm) parle **pidgin nigérian**. On l'ajuste sur plusieurs voix camerounaises de Common Voice pour rapprocher la prononciation. La voix finale est un **mélange** des voix (aucun contributeur n'est reproduit).

1. Le fichier **`KAMA_tts_wes_drive.zip`** doit être dans ton Drive (racine de Mon Drive ou sous-dossier de `KAMA/`). Ne le décompresse pas. Il contient des données Common Voice : Drive privé seulement.
2. **Exécution → Modifier le type d'exécution → GPU T4**.
3. Lance d'abord avec `SMOKE = True` (essai de quelques minutes qui vérifie que tout marche et mesure la vitesse). Si tout passe, mets `SMOKE = False` et relance **depuis la cellule 2**.
4. Si Colab coupe, relance depuis la cellule 2 : rien d'important n'est perdu, seul le modèle final est copié sur Drive à la fin.

Licence : le modèle de base est CC-BY-NC 4.0 (non commercial) ; le modèle ajusté en hérite.

## 1. Réglages

In [ ]:
MON_DRIVE = '/content/drive/MyDrive'
SMOKE = True      # True : 1 passe sur 64 enregistrements, pour vérifier. False : vrai entraînement.
EPOCHS = 30       # passes du vrai entraînement (environ 280 pas par passe)
LOCAL = '/content/KAMA/tts_wes'
REPO = '/content/finetune-hf-vits'
BASE = '/content/mms-tts-pcm-train'     # modèle de base converti pour l'entraînement
SORTIE = '/content/tts_wes_sortie'      # sortie de l'entraînement (disque Colab)
SORTIE_DRIVE = MON_DRIVE + '/KAMA/tts_wes_sorties'   # seuls le modèle final et les échantillons sont copiés ici

## 2. Drive et données
Le zip est décompressé à chaque exécution, sur le disque de Colab.

In [ ]:
import os, glob, shutil, zipfile
from google.colab import drive
drive.mount('/content/drive')
zips = glob.glob(MON_DRIVE + '/KAMA_tts_wes_drive.zip') + glob.glob(MON_DRIVE + '/KAMA/**/KAMA_tts_wes_drive.zip', recursive=True)
assert zips, 'KAMA_tts_wes_drive.zip introuvable dans Mon Drive ni dans KAMA/ : mets-le dans Mon Drive'
print('zip utilisé :', zips[0])
shutil.rmtree(LOCAL, ignore_errors=True)
with zipfile.ZipFile(zips[0]) as z:
    assert z.testzip() is None, 'zip abîmé : renvoie-le depuis ton ordinateur'
    z.extractall('/content')
for f in ('data/train/metadata.csv', 'data/test/metadata.csv', 'data/ecoute_phrases.json'):
    assert os.path.exists(f'{LOCAL}/{f}'), f'manquant dans le zip : {f}'
n_train = sum(1 for _ in open(f'{LOCAL}/data/train/metadata.csv', encoding='utf-8')) - 1
n_test = sum(1 for _ in open(f'{LOCAL}/data/test/metadata.csv', encoding='utf-8')) - 1
print('Données prêtes :', n_train, 'enregistrements d\'entraînement,', n_test, 'de test')

## 3. Installation
Outil d'ajustement de Hugging Face pour MMS-TTS (`ylacombe/finetune-hf-vits`). Les versions sont fixées pour rester proches de celles de l'outil. Si `pip` signale un conflit, redémarre l'exécution (Exécution → Redémarrer) puis relance à partir de cette cellule.

In [ ]:
!pip install -q "transformers==4.44.2" "datasets[audio]==2.21.0" "accelerate==0.34.2" soundfile librosa cython tensorboard
!rm -rf {REPO} && git clone -q https://github.com/ylacombe/finetune-hf-vits {REPO}
%cd {REPO}/monotonic_align
!mkdir -p monotonic_align && python setup.py build_ext --inplace
%cd {REPO}
!nvidia-smi -L

## 4. Modèle de base pour l'entraînement
Convertit le discriminateur de MMS-TTS pcm (nécessaire à l'entraînement, téléchargé depuis `facebook/mms-tts`, environ 560 Mo) et l'assemble avec la voix de base.

In [ ]:
%cd {REPO}
!python convert_original_discriminator_checkpoint.py --language_code pcm --pytorch_dump_folder_path {BASE}
print(os.listdir(BASE))

## 5. Entraînement
En mode `SMOKE`, regarde le temps par pas dans la sortie : durée du vrai entraînement ≈ (enregistrements ÷ 16) × passes × temps par pas.
Après le vrai entraînement, vérifie que les pertes (`loss`) ont baissé et ne sont pas devenues `nan`.

In [ ]:
import json
cfg = {
    'project_name': 'kama_tts_wes', 'push_to_hub': False, 'overwrite_output_dir': True, 'output_dir': SORTIE,
    'dataset_name': f'{LOCAL}/data', 'audio_column_name': 'audio', 'text_column_name': 'text',
    'train_split_name': 'train', 'eval_split_name': 'test',
    'speaker_id_column_name': 'speaker_id', 'override_speaker_embeddings': True,
    'max_duration_in_seconds': 12, 'min_duration_in_seconds': 1.5, 'max_tokens_length': 450,
    'model_name_or_path': BASE, 'preprocessing_num_workers': 2,
    'do_train': True, 'num_train_epochs': 1 if SMOKE else EPOCHS, 'gradient_accumulation_steps': 1, 'gradient_checkpointing': False,
    'per_device_train_batch_size': 16, 'learning_rate': 2e-5, 'adam_beta1': 0.8, 'adam_beta2': 0.99, 'warmup_ratio': 0.01, 'group_by_length': False,
    'do_eval': True, 'eval_steps': 2 if SMOKE else 280, 'per_device_eval_batch_size': 8, 'max_eval_samples': 8 if SMOKE else 40,
    'do_step_schedule_per_epoch': True,
    'weight_disc': 3, 'weight_fmaps': 1, 'weight_gen': 1, 'weight_kl': 1.5, 'weight_duration': 1, 'weight_mel': 35,
    'fp16': True, 'seed': 456, 'report_to': 'tensorboard',
}
if SMOKE:
    cfg['max_train_samples'] = 64
json.dump(cfg, open('/content/config_tts.json', 'w'), indent=1)
os.environ['WANDB_DISABLED'] = 'true'
print('mode', 'ESSAI (SMOKE)' if SMOKE else f'ENTRAINEMENT COMPLET, {EPOCHS} passes')

In [ ]:
%cd {REPO}
!accelerate launch run_vits_finetuning.py /content/config_tts.json

## 6. Voix « mélange »
Le modèle a une voix par contributeur. On ajoute une voix supplémentaire égale à la **moyenne** de toutes : c'est celle que le serveur utilisera, pour qu'aucun contributeur ne soit reproduit. Le numéro à mettre dans `routes.json` est affiché à la fin.

In [ ]:
import torch
from transformers import VitsModel, AutoTokenizer
m = VitsModel.from_pretrained(SORTIE)
tok = AutoTokenizer.from_pretrained(SORTIE)
w = m.embed_speaker.weight.data.clone()
n = w.shape[0]
m.embed_speaker = torch.nn.Embedding(n + 1, w.shape[1])
m.embed_speaker.weight.data = torch.cat([w, w.mean(dim=0, keepdim=True)])
m.config.num_speakers = n + 1
MIX_ID = n
FINAL = '/content/mms-tts-wes-v1'
shutil.rmtree(FINAL, ignore_errors=True)
m.save_pretrained(FINAL)
tok.save_pretrained(FINAL)
print(f'{n} voix + la voix mélange. speaker_id du mélange = {MIX_ID}  <- à mettre dans routes.json (options.speaker_id)')
print(os.listdir(FINAL))

## 7. Échantillons à écouter
Pour chaque phrase : `_base` (voix nigériane d'origine) et `_ajuste` (voix ajustée, mélange). Les phrases viennent des textes validés de la page de test et de phrases de test jamais apprises.

In [ ]:
import re, scipy.io.wavfile as wav
phrases = json.load(open(f'{LOCAL}/data/ecoute_phrases.json', encoding='utf-8'))
vocab = set(json.load(open(f'{FINAL}/vocab.json')))
def propre(t):
    return re.sub(r'\s+', ' ', ''.join(c if c in vocab else ' ' for c in t.lower())).strip()
base = VitsModel.from_pretrained('facebook/mms-tts-pcm').eval()
tok0 = AutoTokenizer.from_pretrained('facebook/mms-tts-pcm')
m.eval()
shutil.rmtree('/content/ecoute', ignore_errors=True)
os.makedirs('/content/ecoute')
for i, t in enumerate(phrases):
    texte = propre(t)
    if not texte:
        continue
    with torch.inference_mode():
        torch.manual_seed(0)
        a = base(**tok0(texte, return_tensors='pt')).waveform[0].numpy()
        torch.manual_seed(0)
        b = m(**tok(texte, return_tensors='pt'), speaker_id=MIX_ID).waveform[0].numpy()
    wav.write(f'/content/ecoute/{i:02d}_base.wav', 16000, a)
    wav.write(f'/content/ecoute/{i:02d}_ajuste.wav', 16000, b)
    print(i, texte)
print(len(os.listdir('/content/ecoute')), 'fichiers dans /content/ecoute')

## 8. Copie sur Drive
Copie `mms-tts-wes-v1.zip` (environ 140 Mo) et `ecoute.zip` dans `Mon Drive/KAMA/tts_wes_sorties`. Télécharge-les : le modèle va dans `KAMA/models/poids/tts/mms-tts-wes-v1` sur ton poste (hors Git).

In [ ]:
suffixe = '_essai' if SMOKE else ''
!cd /content && zip -qr mms-tts-wes-v1{suffixe}.zip mms-tts-wes-v1 && zip -qr ecoute{suffixe}.zip ecoute
os.makedirs(SORTIE_DRIVE, exist_ok=True)
shutil.copy(f'/content/mms-tts-wes-v1{suffixe}.zip', SORTIE_DRIVE)
shutil.copy(f'/content/ecoute{suffixe}.zip', SORTIE_DRIVE)
print(os.listdir(SORTIE_DRIVE))